In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================================
# ADIM 1: FEATURE ENGİNEERİNG (LAG FEATURES DAHİL)
# ============================================================================

def prepare_features_with_lags(df_):
    """
    Tüm feature engineering (lag'ler dahil)
    
    ÖNEMLİ: Bu fonksiyon train+val+test hepsine birlikte uygulanmalı!
    Çünkü lag'ler önceki haftalardan geliyor.
    """
    df = df_.copy()
    
    # Datetime'a çevir
    df['week_start'] = pd.to_datetime(df['week_start'])
    df['customer_created_at'] = pd.to_datetime(df['customer_created_at'])
    
    # SIRALAMA (lag'ler için kritik!)
    df = df.sort_values(['customer_id', 'product_unit_variant_id', 'week_start']).reset_index(drop=True)
    
    print("\n--- FEATURE ENGINEERING (LAG'LER DAHİL) ---")
    
    # === ZAMAN FEATURELARI ===
    df['week'] = df['week_start'].dt.isocalendar().week.astype(int)
    df['month'] = df['week_start'].dt.month.astype(int)
    df['year'] = df['week_start'].dt.year.astype(int)
    df['day_of_year'] = df['week_start'].dt.dayofyear.astype(int)
    
    print("  ✓ Zaman feature'ları oluşturuldu")
    
    # === MÜŞTERİ FEATURELARI ===
    df['customer_age_days'] = (df['week_start'] - df['customer_created_at']).dt.days.clip(lower=0).fillna(0).astype(int)
    
    print("  ✓ Müşteri feature'ları oluşturuldu")
    
    # === O HAFTANIN BİLGİLERİ ===
    # Canlıda bu sütunlarj olmayacak onları 0 kabul et. 
    # Bu bilgileri direkt kullanmayacağız, önceki haftalara dair bilgi almak için bunları kullanıyoruz. 
    if 'qty_this_week' not in df.columns:
        df['qty_this_week'] = 0

    if 'num_orders_week' not in df.columns: 
        df['num_orders_week'] = 0

    if 'spend_this_week' not in df.columns: 
        df['spend_this_week'] = 0
    
    if 'purchased_this_week' not in df.columns: 
        df['purchased_this_week'] = 0

        
    df['qty_this_week'] = df['qty_this_week'].fillna(0).astype(int)
    df['num_orders_week'] = df['num_orders_week'].fillna(0).astype(float)
    df['spend_this_week'] = df['spend_this_week'].fillna(0).astype(int)
    df['purchased_this_week'] = df['purchased_this_week'].fillna(0).astype(float)
    
    
    # =========================================================================
    # LAG FEATURES (ÖNCEKİ HAFTALAR - SADECE GEÇMİŞ!)
    # =========================================================================
    # Sadece shift(1) ile kaydırılmış (geçmiş) bilgileri kullanıyoruz.
    
    group_key = ['customer_id', 'product_unit_variant_id']
    
    # === PURCHASED LAG'LERİ (geçmiş haftalar) ===
    group_purchased = df.groupby(group_key)['purchased_this_week']
    
    df['purchased_lag_1'] = group_purchased.shift(1).fillna(0).astype(int)
    df['purchased_lag_2'] = group_purchased.shift(2).fillna(0).astype(int)
    df['purchased_lag_3'] = group_purchased.shift(3).fillna(0).astype(int)
    df['purchased_lag_4'] = group_purchased.shift(4).fillna(0).astype(int)
    
    print("  ✓ Lag features: purchased_lag_1, purchased_lag_2, purchased_lag_3")
    
    # === ROLLING FEATURES (GEÇMİŞ - bu hafta HARİÇ) ===
    # Son 4 haftada kaç kez alındı? (bu hafta DAHİL DEĞİL)
    
    df['roll_purchased_4_past'] = (
        df['purchased_lag_1'] + df['purchased_lag_2'] + 
        df['purchased_lag_3'] + df['purchased_lag_4']
    )
    
    print("  ✓ Rolling features: roll_purchased_4_past (son 4 hafta, bu hafta hariç)")
    
    # === CUMULATIVE (GEÇMİŞ - geçen haftaya kadar) ===
    # Şimdiye kadar kaç kez alındı? (bu hafta DAHİL DEĞİL)
    df['customer_purch_until_last_week'] = (
        df.groupby(group_key)['purchased_this_week']
        .cumsum()
        .shift(1)  # ← shift(1) ile bu haftayı hariç tutuyoruz
        .fillna(0)
    )
    
    print("  ✓ Cumulative features: customer_purch_until_last_week (geçen haftaya kadar)")
    
    # === QUANTITY LAG'LERİ (geçmiş) ===
    group_quantity = df.groupby(group_key)['qty_this_week']
    
    df['qty_lag_1'] = group_quantity.shift(1).fillna(0).astype(float)
    df['qty_lag_2'] = group_quantity.shift(2).fillna(0).astype(float)
    df['qty_lag_3'] = group_quantity.shift(3).fillna(0).astype(float)
    df['qty_lag_4'] = group_quantity.shift(4).fillna(0).astype(float)
    
    print("  ✓ Quantity lag features: qty_lag_1, qty_lag_2, qty_lag_3")
    
    # === QUANTITY ROLLING (GEÇMİŞ - bu hafta hariç) ===

    df['roll_quantity_4_past'] = (
        df['qty_lag_1'] + df['qty_lag_2'] + 
        df['qty_lag_3'] + df['qty_lag_4']
    )
    
    print("  ✓ Quantity rolling: roll_quantity_4_past (son 4 hafta toplamı, bu hafta hariç)")
    
    # === QUANTITY CUMULATIVE (GEÇMİŞ - geçen haftaya kadar) ===
    df['customer_quantity_until_last_week'] = (
        df.groupby(group_key)['qty_this_week']
        .cumsum()
        .shift(1)  # ← Bu haftayı hariç tut
        .fillna(0)
    )
    
    print("  ✓ Quantity cumulative: customer_quantity_until_last_week (geçen haftaya kadar)")
    

    # =========================================================================
    # BAŞKA ÜRÜNLERLE BİRLİKTE Mİ ALDI? (GEÇMİŞ BİLGİ)
    # =========================================================================
    # purchased_this_week==1 VE num_orders_week > 1 → Başka ürünlerle birlikte aldı
    # Bu bilgiyi shift(1) ile geçmiş haftaya kaydırıp feature yapacağız
    
    df['purchased_with_other_products'] = (
        (df['purchased_this_week'] == 1) & (df['num_orders_week'] > 1)
    ).astype(int)
    
    # Geçmiş haftaya kaydır (lag feature olarak kullan)
    df['purchased_with_others_lag_1'] = (
        df.groupby(['customer_id', 'product_unit_variant_id'])['purchased_with_other_products']
        .shift(1)
        .fillna(0)
        .astype(int)
    )
    
    df['purchased_with_others_lag_2'] = (
        df.groupby(['customer_id', 'product_unit_variant_id'])['purchased_with_other_products']
        .shift(2)
        .fillna(0)
        .astype(int)
    )

    df['purchased_with_others_lag_3'] = (
        df.groupby(['customer_id', 'product_unit_variant_id'])['purchased_with_other_products']
        .shift(3)
        .fillna(0)
        .astype(int)
    )
    
    df['purchased_with_others_lag_4'] = (
        df.groupby(['customer_id', 'product_unit_variant_id'])['purchased_with_other_products']
        .shift(4)
        .fillna(0)
        .astype(int)
    )

    print("  ✓ Birlikte alım features: purchased_with_others_lag_1, purchased_with_others_lag_2, purchased_with_others_lag_3, purchased_with_others_lag_4")


    # =========================================================================
    # ORTALAMA SATIN ALMA ARALIĞI 
    # =========================================================================
    # Müşteri ortalama kaç haftada bir bu ürünü alıyor?
    # Her müşteri-ürün çifti için SABİT bir değer (tüm satırlarda aynı)
    # Ama sadece GEÇMİŞ bilgi kullanır!

    def calculate_avg_purchase_interval_per_customer_product(group):
        """
        Her müşteri-ürün çifti için ortalama satın alma aralığını hesaplar.
        Sadece geçmiş bilgi kullanır: ilk satırı HARİÇ tutar.
        """
        # İlk satır = tahmin yapacağımız satır, onu hariç tut
        # Geri kalan satırlarda purchased_this_week==1 olanları bul
        past_purchases = group.iloc[1:][group.iloc[1:]['purchased_this_week'] == 1]['week_start']
        
        if len(past_purchases) < 2:
            # En az 2 satın alma olmalı ki aralık hesaplayalım
            return 0
        
        # Satın almalar arası gün farklarını hesapla
        intervals = past_purchases.diff().dt.days.dropna()
        
        if len(intervals) > 0:
            # Ortalama aralığı hesapla (hafta cinsinden)
            avg_interval_weeks = intervals.mean() / 7
            return avg_interval_weeks
        else:
            return 0

    # print("  → Ortalama satın alma aralığı hesaplanıyor (müşteri-ürün bazında)...")
# 
    # # Her müşteri-ürün çifti için TEK bir değer hesapla
    # avg_intervals = df.groupby(group_key).apply(calculate_avg_purchase_interval_per_customer_product)
# 
    # # Bu değeri tüm satırlara map et
    # df['avg_purchase_interval_weeks'] = df.set_index(group_key).index.map(avg_intervals).fillna(0).astype(float)
# 
    # print("  ✓ Purchase interval feature: avg_purchase_interval_weeks (müşteri-ürün başına sabit)")


    # =========================================================================
    # SON SATIN ALMADAN BU YANA KAÇINCI HAFTA (RECENCY)
    # =========================================================================

    def weeks_since_last_purchase(group):
        """Her satır için son satın almadan bu yana geçen hafta sayısı"""
        result = []
        
        for i in range(len(group)):
            if i == 0:
                # İlk satırda geçmiş yok
                result.append(999)
            else:
                # Bu satıra kadar purchase olan var mı?
                past_purchases = group.iloc[:i][group.iloc[:i]['purchased_this_week'] == 1]
                
                if len(past_purchases) == 0:
                    # Hiç satın alma olmamış
                    result.append(999)
                else:
                    # Son satın almanın index'inden bu yana kaç hafta geçti
                    last_purchase_week_idx = past_purchases.index[-1]
                    weeks_diff = i - (last_purchase_week_idx - group.index[0])
                    result.append(weeks_diff)
        
        return pd.Series(result, index=group.index)

    print("  → Son satın almadan bu yana geçen hafta hesaplanıyor...")
    df['weeks_since_last_purchase'] = (
        df.groupby(group_key, group_keys=False)
        .apply(weeks_since_last_purchase)
        .fillna(999)
        .astype(int)
    )

    print("  ✓ Recency feature: weeks_since_last_purchase")


    print("\n  NOT: Tüm feature'lar GEÇMİŞ bilgi kullanıyor (shift ile)")
    print("       'Bu haftayı içeren' hiçbir hesaplama YOK!")

    print(f"\n  Toplam feature sayısı: {len(df.columns)}")
        
    return df

In [3]:
# ============================================================================
# ADIM 2: TRAIN-VALIDATION SPLIT (ÇOK ÖNEMLİ!)
# ============================================================================

def split_train_val_test(df, val_weeks=4,test_weeks=4):
    """
    Train setinde:
    - week_start: O haftanın tarihi
    - Target_purchase_next_1w: BİR SONRAKİ hafta alınacak mı?

    - Train: Model eğitimi için
    - Validation: Hiperparametre tuning, model seçimi için
    - Test: Final performans ölçümü için (validation'a overfitting'i önler)
    
    Gerçek test seti (etiketli olmayan) → Canlı/Production
    
    Toplam 46 hafta:
    - Train: 38 hafta (ilk %82)
    - Validation: 4 hafta (sonraki %9)
    - Test: 4 hafta (son %9)
    """
    
    df = df.copy()
    
    # Sadece Target'ı olan satırları kullan (eğer NaN varsa)
    df_with_target = df[
        df['Target_purchase_next_1w'].notna() & 
        df['Target_purchase_next_2w'].notna()
    ].copy()
    
    unique_weeks = np.sort(df_with_target['week_start'].unique())
    
    print("="*80)
    print("TRAIN-VALIDATION-TEST SPLIT (3 PARÇA)")
    print("="*80)
    print(f"\nToplam hafta sayısı: {len(unique_weeks)}")
    print(f"İlk hafta: {unique_weeks[0]}")
    print(f"Son hafta: {unique_weeks[-1]}")
    
    
    # Cutoff'ları hesapla
    test_cutoff = unique_weeks[-test_weeks]  # Son test_weeks hafta → test
    val_cutoff = unique_weeks[-(test_weeks + val_weeks)]  # Ondan önceki val_weeks → validation
    
    print(f"\nSplit stratejisi:")
    print(f"  Validation başlangıcı: {val_cutoff}")
    print(f"  Test başlangıcı: {test_cutoff}")
    
    # Maskeleri oluştur
    train_mask = df_with_target['week_start'] < val_cutoff
    val_mask = (df_with_target['week_start'] >= val_cutoff) & (df_with_target['week_start'] < test_cutoff)
    test_mask = df_with_target['week_start'] >= test_cutoff
    
    df_train = df_with_target[train_mask].copy()
    df_val = df_with_target[val_mask].copy()
    df_test = df_with_target[test_mask].copy()
    
    # İstatistikler
    print(f"\n{'='*80}")
    print("TRAIN SET:")
    print(f"  - Hafta sayısı: {len(df_train['week_start'].unique())}")
    print(f"  - Tarih aralığı: {df_train['week_start'].min()} → {df_train['week_start'].max()}")
    print(f"  - Satır sayısı: {len(df_train):,}")
    print(f"  - Oran: {len(df_train)/len(df_with_target)*100:.1f}%")
    print(f"  - Target 1w pozitif: {df_train['Target_purchase_next_1w'].sum():,} ({df_train['Target_purchase_next_1w'].mean()*100:.2f}%)")
    print(f"  - Target 2w pozitif: {df_train['Target_purchase_next_2w'].sum():,} ({df_train['Target_purchase_next_2w'].mean()*100:.2f}%)")
    
    print(f"\nVALIDATION SET:")
    print(f"  - Hafta sayısı: {len(df_val['week_start'].unique())}")
    print(f"  - Tarih aralığı: {df_val['week_start'].min()} → {df_val['week_start'].max()}")
    print(f"  - Satır sayısı: {len(df_val):,}")
    print(f"  - Oran: {len(df_val)/len(df_with_target)*100:.1f}%")
    print(f"  - Target 1w pozitif: {df_val['Target_purchase_next_1w'].sum():,} ({df_val['Target_purchase_next_1w'].mean()*100:.2f}%)")
    print(f"  - Target 2w pozitif: {df_val['Target_purchase_next_2w'].sum():,} ({df_val['Target_purchase_next_2w'].mean()*100:.2f}%)")
    
    
    print(f"\nTEST SET (Hold-out):")
    print(f"  - Hafta sayısı: {len(df_test['week_start'].unique())}")
    print(f"  - Tarih aralığı: {df_test['week_start'].min()} → {df_test['week_start'].max()}")
    print(f"  - Satır sayısı: {len(df_test):,}")
    print(f"  - Oran: {len(df_test)/len(df_with_target)*100:.1f}%")
    print(f"  - Target 1w pozitif: {df_test['Target_purchase_next_1w'].sum():,} ({df_test['Target_purchase_next_1w'].mean()*100:.2f}%)")
    print(f"  - Target 2w pozitif: {df_test['Target_purchase_next_2w'].sum():,} ({df_test['Target_purchase_next_2w'].mean()*100:.2f}%)")
    
    
    print(f"\n{'='*80}")
    print("NOT:")
    print("  - Train: Modeli eğitmek için")
    print("  - Validation: Hiperparametre optimize etmek için")
    print("  - Test: Final performansı ölçmek için (validation'a overfitting önleme)")
    print("  - Real Test (etiketli olmayan): Canlı tahmin için")
    print(f"{'='*80}")
    
    return df_train, df_val, df_test

In [4]:
# ============================================================================
# ADIM 3: KATEGORİK ENCODE
# ============================================================================

def encode_categorical_features(df_train, df_val, df_test_holdout, df_test_real=None):
    """
    Kategorik değişkenleri encode et
    
    Parametreler:
    - df_train: Eğitim seti
    - df_val: Validation seti
    - df_test_holdout: Test seti (etiketli, bizim ayırdığımız)
    - df_test_real: Gerçek test seti (etiketli olmayan, yarışma verisi)
    """
    
    # Düşük kardinalite → One-Hot
    low_cardinality_cols = ['customer_category', 'customer_status', 'grade_name', 'unit_name']
    low_cardinality_cols = [col for col in low_cardinality_cols if col in df_train.columns]
    
    # Yüksek kardinalite → Label Encoding
    high_cardinality_cols = ['customer_id', 'product_id']
    high_cardinality_cols = [col for col in high_cardinality_cols if col in df_train.columns]
    
    print("\n" + "="*80)
    print("ENCODING STRATEJİSİ")
    print("="*80)
    print(f"One-Hot Encoding: {low_cardinality_cols}")
    print(f"Label Encoding: {high_cardinality_cols}")
    
    # ========================================================================
    # ONE-HOT ENCODING
    # ========================================================================
    
    if low_cardinality_cols:
        print("\n--- ONE-HOT ENCODING ---")
        
        for col in low_cardinality_cols:
            unique_values = df_train[col].astype(str).unique()
            print(f"{col}: {len(unique_values)} değer")
            
            for val in unique_values:
                col_name = f"{col}_{val}"
                df_train[col_name] = (df_train[col].astype(str) == val).astype(int)
                df_val[col_name] = (df_val[col].astype(str) == val).astype(int)
                df_test_holdout[col_name] = (df_test_holdout[col].astype(str) == val).astype(int)
                
                if df_test_real is not None:
                    df_test_real[col_name] = (df_test_real[col].astype(str) == val).astype(int)
    
    # ========================================================================
    # LABEL ENCODING
    # ========================================================================
    
    encoders = {}
    
    if high_cardinality_cols:
        print("\n--- LABEL ENCODING ---")
        
        for col in high_cardinality_cols:
            encoder = LabelEncoder()
            encoder.fit(df_train[col].astype(str))
            
            print(f"{col}: {len(encoder.classes_)} değer")
            
            # Train
            df_train[f'{col}_encoded'] = encoder.transform(df_train[col].astype(str))
            
            # Validation
            df_val[f'{col}_encoded'] = df_val[col].astype(str).map(
                lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1
            )
            
            # Test (hold-out)
            df_test_holdout[f'{col}_encoded'] = df_test_holdout[col].astype(str).map(
                lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1
            )
            
            # Test (real - etiketli olmayan)
            if df_test_real is not None:
                df_test_real[f'{col}_encoded'] = df_test_real[col].astype(str).map(
                    lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1
                )
            
            encoders[col] = encoder
    
    return df_train, df_val, df_test_holdout, df_test_real, encoders

In [5]:
# ============================================================================
# ADIM 4: MODEL EĞİTİMİ
# ============================================================================

def train_model(df_train, df_val, target_column,threshold=0.5):
    """
    Logistic Regression eğitimi
    """
    
    # Feature seçimi
    numeric_features = ['week','month','year','day_of_year','customer_age_days','purchased_lag_1', 'purchased_lag_2', 
    'purchased_lag_3', 'purchased_lag_4','roll_purchased_4_past','customer_purch_until_last_week',
    'qty_lag_1','qty_lag_2','qty_lag_3','qty_lag_4','roll_quantity_4_past','customer_quantity_until_last_week',
    'purchased_with_other_products','purchased_with_others_lag_1','purchased_with_others_lag_2','purchased_with_others_lag_3',
    'purchased_with_others_lag_4','weeks_since_last_purchase']
    
    # One-hot ve label encoded sütunları ekle
    onehot_cols = [col for col in df_train.columns if any(
        col.startswith(f'{cat}_') for cat in ['customer_category', 'customer_status', 'grade_name', 'unit_name']
    )]
    
    label_encoded_cols = [col for col in df_train.columns if col.endswith('_encoded')]
    
    features = numeric_features + onehot_cols + label_encoded_cols
    features = [f for f in features if f in df_train.columns]
    
    print("\n" + "="*80)
    print("MODEL EĞİTİMİ")
    print("="*80)
    print(f"Feature sayısı: {len(features)}")
    print(f"  - Numeric: {len(numeric_features)}")
    print(f"  - One-Hot: {len(onehot_cols)}")
    print(f"  - Label Encoded: {len(label_encoded_cols)}")
    
    # Veri hazırlama
    X_train = df_train[features].fillna(0)
    y_train = df_train[target_column]
    
    X_val = df_val[features].fillna(0)
    y_val = df_val[target_column]
    
    print(f"\nTarget dağılımı:")
    print(f"  Train - Pozitif: {y_train.sum():,} / {len(y_train):,} ({y_train.mean()*100:.2f}%)")
    print(f"  Val   - Pozitif: {y_val.sum():,} / {len(y_val):,} ({y_val.mean()*100:.2f}%)")
    
    # Model
    model = LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced',
        C=0.1  # Regularization
    )
    
    print(f"\nModel eğitiliyor...")
    model.fit(X_train, y_train)
    
    # Tahminler
    y_train_pred_proba = model.predict_proba(X_train)[:, 1]
    y_val_pred_proba = model.predict_proba(X_val)[:, 1]
    
    y_train_pred = (y_train_pred_proba > threshold).astype(int)
    y_val_pred = (y_val_pred_proba > threshold).astype(int)
    
    # Metrikler
    train_auc = roc_auc_score(y_train, y_train_pred_proba)
    val_auc = roc_auc_score(y_val, y_val_pred_proba)
    
    print("\n" + "="*80)
    print("VALİDATİON SONUÇLARI (Model Tuning İçin)")
    print("="*80)
    print(f"Train AUC: {train_auc:.4f}")
    print(f"Val AUC:   {val_auc:.4f}")
    print(f"Fark:      {abs(train_auc - val_auc):.4f}")
    
    if abs(train_auc - val_auc) > 0.05:
        print("  ⚠️  Overfitting var!")
    
    # Confusion Matrix
    cm = confusion_matrix(y_val, y_val_pred)
    print(f"\nValidation Confusion Matrix (threshold={threshold}):")
    print(cm)
    
    return model, features

In [6]:
# ============================================================================
# ADIM 5: TEST (HOLD-OUT) DEĞERLENDİRMESİ
# ============================================================================

def evaluate_on_test(df_test, model, features, target_column, threshold=0.5):
    """
    Hold-out test setinde final performansı ölç
    """
    
    print("\n" + "="*80)
    print("TEST (HOLD-OUT) DEĞERLENDİRMESİ - {target_column}")
    print("="*80)
    print("NOT: Bu skorlar validation'a overfitting'i kontrol eder")
    print("="*80)
    
    X_test = df_test[features].fillna(0)
    y_test = df_test[target_column]
    
    # Tahmin
    y_test_pred_proba = model.predict_proba(X_test)[:, 1]
    y_test_pred = (y_test_pred_proba > threshold).astype(int)
    
    # Metrikler
    test_auc = roc_auc_score(y_test, y_test_pred_proba)
    
    print(f"\nTest AUC: {test_auc:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_test_pred)
    print(f"\nTest Confusion Matrix:")
    print(cm)
    print(f"  - True Negatives:  {cm[0,0]:,}")
    print(f"  - False Positives: {cm[0,1]:,}")
    print(f"  - False Negatives: {cm[1,0]:,}")
    print(f"  - True Positives:  {cm[1,1]:,}")
    
    return test_auc

In [7]:
# ============================================================================
# ADIM 6: GERÇEK TEST (CANLI) TAHMİNİ
# ============================================================================

def predict_real_test(df_test_real, model, features, threshold=0.5):
    """
    Gerçek test seti (etiketli olmayan) için tahmin
    """
    
    print("\n" + "="*80)
    print("GERÇEK TEST (CANLI) TAHMİNİ")
    print("="*80)
    
    X_test_real = df_test_real[features].fillna(0)
    
    # Tahmin
    predictions_proba = model.predict_proba(X_test_real)[:, 1]
    predictions_binary = (predictions_proba > threshold).astype(int)
    
    print(f"  Tahmin sayısı: {len(predictions_proba):,}")
    print(f"  Pozitif tahmin: {predictions_binary.sum():,} ({predictions_binary.mean()*100:.2f}%)")
    print(f"  Ortalama olasılık: {predictions_proba.mean():.4f}")
    
    return predictions_proba, predictions_binary

In [8]:
# ============================================================================
# ANA PİPELİNE
# ============================================================================

def main_pipeline(df_train_raw, df_test_real_raw, val_weeks=4, test_weeks=4, threshold=0.5):
    """
    Tam pipeline:
    1. df_train_raw → Train + Val + Test (hold-out)
    2. df_test_real_raw → Canlı tahmin
    """
    
    print("="*80)
    print("ZAMAN SERİSİ TAHMİN PİPELİNE")
    print("="*80)
    
    # 1. Feature Engineering
    print("\n[1/7] Feature Engineering...")
    df_train = prepare_features_with_lags(df_train_raw)
    df_test_real = prepare_features_with_lags(df_test_real_raw)
    
    # 2. Train-Val-Test Split (3 parça!)
    print("\n[2/7] Train-Validation-Test Split...")
    df_train_split, df_val_split, df_test_holdout = split_train_val_test(
        df_train, val_weeks=val_weeks, test_weeks=test_weeks
    )
    
    # 3. Encoding
    print("\n[3/7] Kategorik encoding...")
    df_train_split, df_val_split, df_test_holdout, df_test_real, encoders = encode_categorical_features(
        df_train_split, df_val_split, df_test_holdout, df_test_real
    )
    
    # ==========================================================================
    # 1 HAFTA SONRASI MODELİ
    # ==========================================================================
    print("\n" + "="*80)
    print("1 HAFTA SONRASI (Target_purchase_next_1w)")
    print("="*80)
    
    print("\n[4a/7] Model eğitimi (1w)...")
    model_1w, features_1w = train_model(
        df_train_split, df_val_split, 
        target_column='Target_purchase_next_1w',
        threshold=threshold
    )
    
    print("\n[5a/7] Test değerlendirmesi (1w)...")
    test_auc_1w = evaluate_on_test(
        df_test_holdout, model_1w, features_1w,
        target_column='Target_purchase_next_1w',
        threshold=threshold
    )
    
    print("\n[6a/7] Gerçek test tahmini (1w)...")
    pred_proba_1w, pred_binary_1w = predict_real_test(
        df_test_real, model_1w, features_1w, threshold=threshold
    )
    
    # ==========================================================================
    # 2 HAFTA SONRASI MODELİ
    # ==========================================================================
    print("\n" + "="*80)
    print("2 HAFTA SONRASI (Target_purchase_next_2w)")
    print("="*80)
    
    print("\n[4b/7] Model eğitimi (2w)...")
    model_2w, features_2w = train_model(
        df_train_split, df_val_split,
        target_column='Target_purchase_next_2w',
        threshold=threshold
    )
    
    print("\n[5b/7] Test değerlendirmesi (2w)...")
    test_auc_2w = evaluate_on_test(
        df_test_holdout, model_2w, features_2w,
        target_column='Target_purchase_next_2w',
        threshold=threshold
    )
    
    print("\n[6b/7] Gerçek test tahmini (2w)...")
    pred_proba_2w, pred_binary_2w = predict_real_test(
        df_test_real, model_2w, features_2w, threshold=threshold
    )
    
    # ==========================================================================
    # SUBMİSSİON
    # ==========================================================================
    print("\n[7/7] Submission hazırlanıyor...")
    submission = pd.DataFrame({
        'ID': df_test_real['ID'],
        'Target_purchase_next_1w': pred_proba_1w,
        'Target_purchase_next_2w': pred_proba_2w
    })
    
    # ==========================================================================
    # ÖZET
    # ==========================================================================
    print("\n" + "="*80)
    print("ÖZET")
    print("="*80)
    print(f"1w Model:")
    print(f"  Test AUC: {test_auc_1w:.4f}")
    print(f"  Pozitif tahmin: {pred_binary_1w.sum():,} / {len(pred_binary_1w):,} ({pred_binary_1w.mean()*100:.2f}%)")
    
    print(f"\n2w Model:")
    print(f"  Test AUC: {test_auc_2w:.4f}")
    print(f"  Pozitif tahmin: {pred_binary_2w.sum():,} / {len(pred_binary_2w):,} ({pred_binary_2w.mean()*100:.2f}%)")
    
    print(f"\nSubmission hazır: {len(submission):,} satır")
    print(f"Sütunlar: {list(submission.columns)}")
    
    return {
        'model_1w': model_1w,
        'model_2w': model_2w,
        'features_1w': features_1w,
        'features_2w': features_2w,
        'submission': submission,
        'test_auc_1w': test_auc_1w,
        'test_auc_2w': test_auc_2w
    }


In [9]:
# ============================================================================
# KULLANIM
# ============================================================================

# Veriyi yükle
df_train_raw = pd.read_csv("../data/train.csv")
df_test_real_raw = pd.read_csv("../data/test.csv")

# Pipeline çalıştır
results = main_pipeline(
    df_train_raw, 
    df_test_real_raw,
    val_weeks=4,
    test_weeks=4,
    threshold=0.5
)

# Submission kaydet
results['submission'].to_csv('submission_purchase.csv', index=False)

print("\n✅ Submission kaydedildi: submission.csv")
print(f"   Boyut: {len(results['submission']):,} satır x {len(results['submission'].columns)} sütun")


ZAMAN SERİSİ TAHMİN PİPELİNE

[1/7] Feature Engineering...

--- FEATURE ENGINEERING (LAG'LER DAHİL) ---
  ✓ Zaman feature'ları oluşturuldu
  ✓ Müşteri feature'ları oluşturuldu
  ✓ Lag features: purchased_lag_1, purchased_lag_2, purchased_lag_3
  ✓ Rolling features: roll_purchased_4_past (son 4 hafta, bu hafta hariç)
  ✓ Cumulative features: customer_purch_until_last_week (geçen haftaya kadar)
  ✓ Quantity lag features: qty_lag_1, qty_lag_2, qty_lag_3
  ✓ Quantity rolling: roll_quantity_4_past (son 4 hafta toplamı, bu hafta hariç)
  ✓ Quantity cumulative: customer_quantity_until_last_week (geçen haftaya kadar)
  ✓ Birlikte alım features: purchased_with_others_lag_1, purchased_with_others_lag_2, purchased_with_others_lag_3, purchased_with_others_lag_4
  → Son satın almadan bu yana geçen hafta hesaplanıyor...
  ✓ Recency feature: weeks_since_last_purchase

  NOT: Tüm feature'lar GEÇMİŞ bilgi kullanıyor (shift ile)
       'Bu haftayı içeren' hiçbir hesaplama YOK!

  Toplam feature sayısı: 

In [11]:
df_purchase = pd.read_csv("submission_purchase.csv")

df_quantity = pd.read_csv("submission_quantity.csv")

In [14]:
df_result = df_purchase.merge(df_quantity,on='ID')[['ID','Target_purchase_next_1w','Target_qty_next_1w','Target_purchase_next_2w','Target_qty_next_2w']]

In [15]:
df_result.to_csv("result_base_model.csv")